# Key-Value Stores (Redis)


# Introduction:

### Redis:
   <a href="https://redis.io/">Redis</a> Redis is an open source (BSD licensed), in-memory data structure store, used as a database, cache and message broker. It supports data structures such as strings, hashes, lists, sets, sorted sets with range queries, bitmaps, hyperloglogs, geospatial indexes with radius queries and streams.<br/>
   
<img src="https://upload.wikimedia.org/wikipedia/fr/thumb/6/6b/Redis_Logo.svg/640px-Redis_Logo.svg.png" width ="250" >


#### <a href='https://redislabs.com/redis-enterprise/data-structures/'>Redis Data Structures</a>
* Redis is not a plain key-value store, it is actually a data structures server, supporting different kinds of values.
* An introduction to Redis data types and abstractions https://redis.io/topics/data-types-intro
* Redis keys are always strings.


<img src='https://redis.io/wp-content/uploads/2021/07/key-value-data-stores-2-v2-920x612-1.png' width='500' >

### How To Query Redis!

- Commands for each data type for common access patterns, with bulk operations, and partial transaction support.

### PreLab

#### 1. Install Redis on Windows
- Redis is a cross-platform DB, We can install it on Linux, or Windows, ..etc.
- There are two ways to install Redis under Windows
    - Download the latest Redis .msi file from https://github.com/MSOpenTech/redis/r... and install it. 
    
    - You can choose either from these sources
        - https://github.com/microsoftarchive/redis/releases or
        - https://github.com/rgl/redis/downloads

- Personally I prepared the first option
- Download Redis-x64-2.8.2104.zip
- Extract the zip to the prepared directory
- Run redis-server.exe
- Run redis-cli.exe
- For more info follow this setup-video tutorial (https://www.youtube.com/watch?v=188Fy-oCw4w)


#### Linux and Debian 

- Even quicker and dirtier instructions for Debian-based Linux distributions are as follows:
    - download Redis from http://redis.io/download 
    - extract, run make && sudo make install
    - Then run sudo python -m easy_install redis hiredis (hiredis is an optional performance-improving C library).

#### 2. Install the Python Package ("<a href='https://pypi.org/project/redis/'>redis</a>") to connecto to Redis 
- use th command ```pip install redis``` in your command line.


#### (more) Accessing Redis from Command Line:
- Add the Redis installation "/home" and "/bin" directories to the environment variables.
- start Redis server in one command window(CMD, poweshell, ..etc)using the command ```redis-server```.
- In another command window, start your Redis Client using the command ```redis-cli```
- Now you have the Redis Client Shell connected to the default <b>db0</b> DB. 

In [ ]:
import redis

In [ ]:
import pandas as pd

In [ ]:
from pprint import pprint
from time import sleep

import warnings
warnings.filterwarnings('ignore')

##### Get a client connection to redis server, using the url and the port, and Db

In [ ]:
r = redis.Redis(host='redis', port=6379, db=0)

## Task 0: First Steps in Redis

### Demo the string keys

In [ ]:
r.set('language','Python')

##### Key expiration (e.g Think about Sessions management )


- By default, keys are retained, but we can make our keys (data) vanish after a specified time.
- This can be set while creating the key, or for already existing keys.

In [ ]:
## use expire(key,time in_secs), after this time key will vanish
print(r.expire('language', 6))
#ttl(key) time_to_live, checks remaining time to live! 
print(r.ttl('language'))
sleep(3)
print(r.ttl('language'))

#### Check if the key already expired!
- Use exists(key) function

In [ ]:
###YOUR CODE HERE

#### Setting multiple String Keys, values 

In [ ]:
r.mset({"Croatia": "Zagreb", "Bahamas": "Nassau"})

#### Get the value of the key 'Croatia'

In [ ]:
###YOUR CODE HERE

#### Set String as JSON value 

In [ ]:
r.set('myJsonData' , '{"name": "Ragab", "age": 40}')

#### Get the previous JSON value

In [ ]:
###YOUR CODE HERE

#### Rename the key 'myjsonData' into 'myJsonInfo'

In [ ]:
r.rename('myJsonData', 'myJsonInfo')

#### Delete 'myJsonInfo' key-value pair.

In [ ]:
###YOUR CODE HERE

#### Check if it's deleted already !

In [ ]:
# we try to get the value of that key!
print(r.get('myJsonInfo'))

#or we can check if it's not existing any more !
print(r.exists ('myJsonInfo'))

### Demo The Lists

- Think of Lists as ordered sequence of strings like java ArrayList, javasrcript array, or python n lists.
- We can use lists to implement stacks and queues.
    - If you need a **Queue**, just use **RPUSH** and **LPOP**.
    - If you need a **Stack**, just use **RPUSH** and **RPOP**.
- Lists can **accept duplicates**.
- A single Resid List can hold over **4B** entries!!


#### Create a List of customers, and add elements to it!

In [ ]:
r.lpush('customers','Ragab')

This will return "1" meaning the list now contains only one element

In [ ]:
r.lpush('customers','Riccardo')

In [ ]:
r.lpush('customers','Riccardo')

#### Get current memebrs of the list
- Hint: use <code>lrange</code> function

In [ ]:
###YOUR CODE HERE

* We can clearly notice that **LPUSH** function/command adds elements to the left of the list.
* and that Lists also **accept duplicate** items.

#### Adding element to the Right of the customers list
- Add customer 'Kim' to the right of the list
- Use <code>rpush</code> function

In [ ]:
###YOUR CODE HERE

In [ ]:
#check the list elements again
r.lrange('customers',0,-1)

#### Insert 'Jan' between Riccardo and Ragab

In [ ]:
r.linsert('customers','BEFORE','Ragab','Jan')

In [ ]:
r.lrange('customers',0,-1)

#### Get only the first element 

In [ ]:
###YOUR CODE HERE

#### Get only the first 3 elements  

In [ ]:
r.lrange('customers',0,2)

#### Get the Length of the List
- Use <code>llen</code> function

In [ ]:
###YOUR CODE HERE

#### Delete the first element on the left

In [ ]:
r.lpop('customers')

In [ ]:
#check the list elements again
r.lrange('customers',0,-1)

#### Delete the first element on the right

In [ ]:
###YOUR CODE HERE

In [ ]:
#check the list elements again
r.lrange('customers',0,-1)

#### Notes on the performance of Lists:
- **LPOP**, **LPUSH**, and **LLEN** commands are all **O(n)** cosnstant time operations. 
    - Their performance is independent of the lenght of the list.
- **LRANGE** is **O(s+n)**, such that **s** is the distance of the start offest from the head, and **n** is the number of the elements in the specified range.
    - Thus, we need to be careful with LRANGE especially with extra long lists, or when we retrieve thousands or more elements!!
<img src='figs/ListsPerformance.JPG' width= '200'>

### Demo The Sets

- **Unordered** collection of strings.
- Contains **no duplicates**.
    - This makes Sets supernatual option for **de-dupication** applications.
- Questions that we can Answer using Sets:
    - **Did I see this IP address in the last hour?**
    - **Is this user online?**
    - **Has this URL been balcklisted?**
- All of these questions can be answered in **O(1)** time.
- Sets support standard operations:
    - **Intersection** 
    - **Difference**
    - **Union**

#### Create a Set of online players with the key "players:online" ["Riccardo", and "Ragab"]

In [ ]:
r.sadd('players:online',"Riccardo","Ragab")

#### Try to another Online-player "Ragab" in the Set
- Write down what did you noticed?!

In [ ]:
###YOUR CODE HERE

#### Check if the Set of Online Players contain the player "Riccardo"

In [ ]:
r.sismember('players:online',"Riccardo")

#### Check if the Set of Online Players contain the player "Fabiano"

In [ ]:
r.sismember('players:online',"Fabiano")

#### Create Another Set with the key-name ("Friends") that has ["Riccardo", "Fabiano", "Hassan"]

In [ ]:
###YOUR CODE HERE

#### Get the two lists memebers  

In [ ]:
print ("FirstSet:" ,r.smembers('players:online'))
print ("SecondSet:",r.smembers('friends'))

#### Get the intersection of these two sets 

In [ ]:
#Intersction
print(r.sinter('players:online','friends'))

#### Get the Union of these two sets 

In [ ]:
###YOUR CODE HERE

#### Get the Length of the two Sets 

In [ ]:
#Length
print(r.scard('friends'))
print(r.scard('players:online'))

#### Move "Fabiano" to the Online Players Set

In [ ]:
r.smove('friends', 'players','Fabiano')

#### Get the Length of the two Sets After this move!

In [ ]:
###YOUR CODE HERE

#### Remove "Ragab" from the players Set, and show the palyers Set after this removal

In [ ]:
r.srem('players:online',"Ragab")
r.smembers('players:online')

### Demo The SORTED SETS
- REDIS sorted sets are **ordered** collections of unique members.
- These memebrs are ordered according to their **asociated score**.
- Whenever you add to the sorted set, you are specifying a **memeber** and a **score**.
- Sorted Sets keep every thing sorted from the begininng.
- Sorted sets are good choice for:
    - **priority queues**
    - **Low-latency leaderboards**
    - **Secondary indexing**
- Questions that we can Answer using Sorted Sets (e.g, in an online-game ):
    - **Who are the top 10 players?
    - **what is the rank of a specific Player?
    - **what is the current score of the player?

#### Let's Create our Leaderboard
- In the scenario of online game, each player will have a score of '**experience**' achieving some tasks/goals,..etc.

#### Initially, Let's give a score of 0 experience to all of our players:
- We have three Players ("Ragab", "Fabianno", and "Riccardo")

In [ ]:
r.zadd('players:exp',{'Ragab':0})
r.zadd('players:exp',{'Riccardo':0})
r.zadd('players:exp',{'Fabiano':0})

#### Increment the experiernce score of our players
Let's pretend that our players have copleted some missions and they got these experience points 40,60,80 for "Ragab", "Riccardo", and "Fabiano" respectively

In [ ]:
print(r.zincrby('players:exp',40,'Ragab'))
print(r.zincrby('players:exp',60,'Riccardo'))
print(r.zincrby('players:exp',80,'Fabiano'))

#### Let's Punish one of the players penalizing him with 5 points of experience

In [ ]:
###YOUR CODE HERE

#### Get the Top 3 players in our game

In [ ]:
r.zrevrange('players:exp',0,2)

#### GET the Top 3 players in our game showing their scores

In [ ]:
###YOUR CODE HERE

#### Get the Ranak of the players "Ragab", and "Fabiano"
- Look at the difference between **Zrank**, and **zrevrank**

In [ ]:
print(r.zrevrank('players:exp', 'Ragab'))
print(r.zrevrank('players:exp', 'Fabiano'))

#### Get the score of the player "Riccardo"

In [ ]:
print (r.zscore('players:exp', 'Riccardo'))

### Demo the HASHES 
- Hashes are one of the most useful Redis data structures.
- Hashes are collections of field-value pairs.

#### Let's Create a Hash of Players in an online game

- Each player has the following fields:
    - NAME
    - RACE
    - LEVEL
    - HEALTH
    - GOLD

#### Let's Create our first player ('player:101')

In [ ]:
r.hset('player:101','name','Cyclops')
r.hset('player:101','race','Elf')
r.hset('player:101','level',4)
r.hset('player:101','health',20)
r.hset('player:101','gold',500)

#### Adding another player (player:102)
- We can use <code>HMSET</code> value pairs to the Hash.

In [ ]:
player2 = {"name":"Wolverine", 
 "race":"Elf", 
 "level":6, 
 "health":200, 
 "gold":4000}
#We use HMSET for ading multi-field value pairs to the Hash
r.hmset('player:102', player2)

#### Get the information of the player Hashes ('player:101', 'player:102' )

In [ ]:
pprint(r.hgetall('player:101'))
print("\n")
pprint(r.hgetall('player:102'))

#### Get the **name** of the second player
- Use <code>hget(Hash_key,field)</code>

In [ ]:
###YOUR CODE HERE

#### Get the name, level and the race of the second player
- Use <code>hmget(Hash_key,field1,field2,..)</code>

In [ ]:
###YOUR CODE HERE

#### updating the Hash with adding a new field
- For player ('player:101'), add the **status** as '**Killed**' 

In [ ]:
###YOUR CODE HERE

#### Check if added field 'status' to the first player ('player:101')

In [ ]:
pprint(r.hgetall('player:101'))

#### updating the Hash with deleting the  'status' field for 'player:101'

In [ ]:
r.hdel('player:101','status')

#### Check if the  field 'status' is deleted from the first player ('player:101')

In [ ]:
pprint(r.hgetall('player:101'))

#### In such games, players recieve gold points, after completing objectives, or defeating enemies.
- Let's add some gold points to player:102 "Wolverine", Increase him by 25 points. 

In [ ]:
print("Gold Before: ")
pprint(r.hget('player:102', 'gold'))

r.hincrby('player:102', 'gold', 25)

print("\nAfter: ")
pprint(r.hget('player:102', 'gold'))

**Notes on incrementing & decrementing Hash values:** 
- HINCRBY still operates on a hash value that is a string, but it tries to interpret the string as a **base-10 64-bit signed integer** to execute the operation.

- This applies to other commands related to incrementing and decrementing other data structures, namely **INCR**, **INCRBY**, **INCRBYFLOAT**, **ZINCRBY**, and **HINCRBYFLOAT**.

- You’ll get an error if the string at the value can’t be represented as an integer.


#### Notes on the performance of Redis Hashes:
- HGET, HSET, HINCRBY, HDEL are O(1) constant time opeations regardless of the size of the Hash.
- Whreas, HGETALL is O(n),with n being the number of fields in the Hash.
    - In Big Hashes of Thousands of fields, it's usually more effiecient to specify the fields you want, rather than retreiving all of the fields.
    
<img src='figs/HashPerformance.JPG' width= '200'>